# Manual Day — Neural Networks

Closed-agent, ~60 min. **Two problems — work through them in order; complete as many as time permits.** Complete every cell marked `# FILL IN`, then run the check cell.

## Mini-lecture: what a training loop is

Everything in machine learning that "learns" is one loop. You have a model with
**parameters**, a single number that says how wrong the model currently is, and a
rule for nudging the parameters to make that number smaller. Then you repeat.

Four steps per pass:

1. **Forward.** Push the inputs through the model with the current parameters to
   get predictions.
2. **Loss.** Compare predictions to targets and collapse that comparison into one
   number $L$. Smaller is better, by construction — you choose $L$ so that it is.
3. **Gradient.** Compute $\partial L/\partial\theta$ for each parameter $\theta$:
   which way is uphill, and how steeply.
4. **Update.** Step downhill,
   $\theta \leftarrow \theta - \eta\,\partial L/\partial\theta$, where the
   **learning rate** $\eta$ sets the stride.

Stripped of anything specific, that is:

```python
for epoch in range(num_epochs):
    pred   = model(x, params)        # 1. forward
    L      = loss(pred, y)           # 2. loss
    g      = gradient_of_L(params)   # 3. gradient
    params = params - lr * g         # 4. update
```

A deep neural network differs from today's straight line *only* in how complicated
step 1 is. Steps 2 through 4 are identical, which is why this loop is worth knowing
cold before anything fancier.

Vocabulary we will use all semester:

- **epoch** — one pass through the data. Our dataset is 25 points and every pass
  uses all of them, so here one epoch is exactly one gradient step. On large data
  you step on random subsets instead (**mini-batches**), which is where the
  *stochastic* in stochastic gradient descent comes from.
- **learning rate** $\eta$ — the stride length. Too small and you crawl; too large
  and you overshoot, so the loss oscillates or climbs. It is given to you below as
  `lr = 0.1`.
- **convergence** — the loss stops falling. That does *not* by itself mean you
  found the right answer: it may be the noise floor of the data (which is what
  happens below), a step size too large to settle, or a bad minimum.

**How to tell it is working.** Print the loss every few epochs and watch it fall.
If it rises, your gradient has the wrong sign or your learning rate is too big. If
it never moves, your gradient is zero — or, the classic, you computed the update
and never assigned it back to the parameters.

Finally, notice the shape: advance a state by a local rule, stop when a number
stops moving. That is the same skeleton as the power iteration you wrote last time,
as Euler integration and Jacobi relaxation in week 4, and as Metropolis later on.
Only the update line changes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)          # reproducible data for grading

# --- data: noise around the line  y = 10*x + 4  (lifted from lecture) ---
num_points = 25
xs = np.random.normal(size=(num_points, 1))
noise = np.random.normal(scale=.5, size=(num_points, 1))
ys = 10 * xs + 4 + noise

# --- hyperparameters (GIVEN) ---
m, b = 0.0, 0.0            # parameters, initialized at the origin
lr = 0.1                   # learning rate
num_epochs = 50            # gradient-descent steps

plt.scatter(xs, ys); plt.xlabel("x"); plt.ylabel("y"); plt.title("training data");

## Problem 1 — Linear regression by hand-derived gradient

## The problem

The simplest possible neural network is a straight line, $f(x) = m\,x + b$, a
function with two parameters. We fit it by **gradient descent** on the
mean-squared error

$$L(m,b) = \frac{1}{N}\sum_i \big(y_i - (m x_i + b)\big)^2 .$$

Differentiate it by hand — this is the whole point, **no autodiff, no jax** — to
get $\partial L/\partial m$ and $\partial L/\partial b$, then descend:

$$m \leftarrow m - \eta\,\frac{\partial L}{\partial m},
\qquad
b \leftarrow b - \eta\,\frac{\partial L}{\partial b}.$$

Not sure your derivation is right? That is what Problem 2 is for — it checks these
derivatives numerically. If it fails, come back and fix this one.

**Known answer:** the data was generated from $y = 10x + 4$, so after training
you should recover $m \approx 10$ and $b \approx 4$, with the MSE decreasing
monotonically toward the noise floor.

In [ ]:
# FILL IN: predictions, your two hand-derived gradients, and the descent step
losses = []
for epoch in range(num_epochs + 1):
    f = ...        # predictions of the current line
    dLdm = ...     # dL/dm
    dLdb = ...     # dL/db
    # ... then step m and b downhill, using the learning rate lr

    losses.append(np.mean((ys - f) ** 2))
    if epoch % 10 == 0:
        print(f"epoch {epoch:2d}: MSE={losses[-1]:.4f}  m={m:.3f}  b={b:.3f}")

In [ ]:
# --- CHECK (given): do not edit ---
print(f"final parameters:  m = {m:.3f},  b = {b:.3f}")
assert abs(m - 10.0) < 1.0, f"expected m ~ 10, got {m:.3f}"
assert abs(b - 4.0)  < 1.0, f"expected b ~ 4,  got {b:.3f}"

# MSE must be trending down (compare start vs end)
assert losses[-1] < losses[0], "MSE did not decrease"
print(f"MSE trend:  {losses[0]:.3f}  ->  {losses[-1]:.3f}   (decreasing: OK)")

# fit vs data
xline = np.linspace(xs.min(), xs.max(), 100).reshape(-1, 1)
plt.scatter(xs, ys, label="data")
plt.plot(xline, m * xline + b, "r-", label=f"fit: {m:.2f}x + {b:.2f}")
plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.title("fit vs data")
print("All checks passed.")

## Problem 2 — Gradient check by finite differences

Autodiff is convenient, but you should be able to confirm a gradient is correct
without it. A model-agnostic sanity check is the **central finite difference**:
perturb one parameter by a small step $h$ and measure how the loss responds,

$$\frac{\partial L}{\partial m} \approx \frac{L(m+h,\,b) - L(m-h,\,b)}{2h},
\qquad
\frac{\partial L}{\partial b} \approx \frac{L(m,\,b+h) - L(m,\,b-h)}{2h}.$$

Using the same data $y = 10x + 4 + \text{noise}$ and the loss
$L(m,b) = \text{mean}\big((y-(mx+b))^2\big)$, estimate the gradient numerically
at the test point $(m,b) = (3.0,\,1.0)$ with $h = 10^{-5}$.

**Known answer:** the numeric gradient must match the analytic gradient **you
derived in Problem 1**, to a relative error below $10^{-4}$.

In [ ]:
# FILL IN: central finite-difference estimate of dL/dm and dL/db at (m,b)=(3.0,1.0)
m0, b0 = 3.0, 1.0     # test point (GIVEN)
h = 1e-5              # finite-difference step (GIVEN)

# Write the loss as a function of (m, b), then perturb one argument at a time.

num_dLdm = ...        # numeric dL/dm
num_dLdb = ...        # numeric dL/db

print(f"numeric gradient:  dL/dm = {num_dLdm:.6f},  dL/db = {num_dLdb:.6f}")

In [ ]:
# --- CHECK (given): do not edit ---
f = m0 * xs + b0
ana_dLdm = -2 * np.mean((ys - f) * xs)    # analytic dL/dm at the test point
ana_dLdb = -2 * np.mean(ys - f)           # analytic dL/db at the test point
print(f"analytic gradient: dL/dm = {ana_dLdm:.6f},  dL/db = {ana_dLdb:.6f}")

rel_m = abs(num_dLdm - ana_dLdm) / abs(ana_dLdm)
rel_b = abs(num_dLdb - ana_dLdb) / abs(ana_dLdb)
print(f"relative error:    dL/dm: {rel_m:.2e},  dL/db: {rel_b:.2e}")

assert rel_m < 1e-4, f"dL/dm off: rel error {rel_m:.2e}"
assert rel_b < 1e-4, f"dL/db off: rel error {rel_b:.2e}"
print("All checks passed.")